# Profit-based beer game — full simulation

1. **Objective: maximize total supply-chain profit**, not simply minimize inventory. Revenue is earned on actual customer deliveries. Factory production, holding inventory, and backlogs have costs.
2. **Selected run: API / full game / 8 groups per cell / 36 weeks.** That is 32 four-role groups, 128 agents, and **4,608 ordering decisions**, plus 128 rules checks. This is not another 48-order screen.
3. Each role starts with **4 units on hand** (previously 12); external demand starts at **8/week** and later varies around **8, 12, or 16**. Initial shipping/production flow remains 4, so replenishment matters early.
4. No positive minimum order is imposed. Zero orders remain valid in surplus states. More orders are not automatically better: committed factory production is paid even when goods are not sold.
5. The free checks below verify profit accounting, known-beta recovery, and policy incentives before paid calls. They do not establish how GPT or students will behave.
6. This is a **new profit-based variant**, not an exact replication of the original cost-only beer game. Do not pool it with earlier trials. H2_API remains low/high reasoning effort, not human 60/120-second time pressure.

**To verify free:** set MODE = "offline", restart, and Run All. **To run GPT:** use MODE = "api". Your $3 local spending guard is retained and may stop well before completion; review cost scenarios. No previous results are overwritten. Keep this notebook beside beer_game_v3.py.


In [ ]:
# Uncomment only if dependencies are missing, then restart the kernel.
# %pip install "openai>=2,<3" "pydantic>=2.8,<3" numpy pandas scipy matplotlib ipykernel
from pathlib import Path
import sys, importlib, os, getpass

BASE = Path.cwd()
if (BASE / "Project" / "beer_game_experiment").is_dir():
    BASE = BASE / "Project" / "beer_game_experiment"
elif (BASE / "beer_game_experiment").is_dir():
    BASE = BASE / "beer_game_experiment"
if not (BASE / "beer_game_v3.py").exists():
    raise FileNotFoundError("Open this notebook in the beer_game_experiment folder, beside beer_game_v3.py.")
sys.path.insert(0, str(BASE))
import beer_game_v3 as study
importlib.reload(study)
from IPython.display import display, Markdown


In [ ]:
# View the ORIGINAL completed API game now — this cell sends no requests.
import pandas as pd
original_run = BASE / "results" / "api_023f89662165ca64"
if (original_run / "performance.csv").exists():
    old_performance = pd.read_csv(original_run / "performance.csv")
    old_teams = old_performance.groupby(["group_id", "trained", "deadline_s"]).total_cost.sum().reset_index()
    display(Markdown("### Original API game: supply-chain cost (game units, not API dollars)"))
    display(old_teams.groupby(["trained", "deadline_s"]).total_cost.agg(["count", "mean", "median", "min", "max"]))
    old_beta = pd.read_csv(original_run / "agent_beta_fits.csv")
    old_beta["usable_beta"] = old_beta.beta.where(old_beta.usable)
    display(Markdown("### Original API game: β fits (blank = unidentified; timing exclusions are separate)"))
    display(old_beta[["agent_id", "alpha", "usable_beta", "usable", "primary_eligible", "fit_flag"]])
    print("Full prior tables:", original_run / "performance.csv", original_run / "agent_beta_fits.csv")


## 1. Full-game settings

The selected settings run **32 groups / 128 agents / 4,608 orders + 128 rules checks**. A $3 local allowance can stop before completion; it is not the projected cost or your account balance. Review the printed cost scenarios. Raise only BUDGET_USD to resume after a spending stop.

Use a fresh REPLICATE_ID for independent data. Successful cached responses are reused only for the same protocol. Changing the engine code or experimental settings creates a fresh run. Do not increase group count repeatedly while checking significance. Select a confirmation sample size after measuring estimation reliability and a meaningful effect size.


In [ ]:
MODE = "api"                 # "offline" or "api"
STAGE = "game"                   # Full four-role supply chain, not another diagnostic screen
GROUPS_PER_CELL = 8               # Reporting floor, NOT a power guarantee. Fix sample size before running.
WEEKS = 36
REPLICATE_ID = "v3-profit-01"
SEED = 20260927
BUDGET_USD = 3.00                 # Local estimate guard for THIS run, not your account balance
MAX_OUTPUT_TOKENS = 8192          # Includes reasoning; same ceiling in all cells
REQUEST_TIMEOUT_S = 180           # Infrastructure timeout, NOT an experimental time treatment

CONFIG = dict(mode=MODE, stage=STAGE, model=study.MODEL,
    groups_per_cell=GROUPS_PER_CELL, weeks=WEEKS, replicate_id=REPLICATE_ID,
    seed=SEED, budget_usd=BUDGET_USD, max_output_tokens=MAX_OUTPUT_TOKENS,
    timeout_s=REQUEST_TIMEOUT_S)
requests_planned = 4*(4+len(study.diagnostic_states())) if STAGE == "diagnostics" else GROUPS_PER_CELL*16*(WEEKS+1)
print("Model:", study.MODEL)
print(f"{MODE.upper()} / {STAGE}: {requests_planned:,} planned requests; {BUDGET_USD:.2f} USD local allowance.")
print("FULL GAME selected" if STAGE == "game" else "DIAGNOSTICS ONLY: no game performance or beta")
print("Offline mode sends zero requests. API mode is paid.")
# Scenarios, not forecasts: cache discounts ignored. Game histories can be longer.
for output_tokens in (200, 1000, 3000):
    estimate = requests_planned*(2000*study.RATES['input']+output_tokens*study.RATES['output'])/1e6
    print(f"If average input=2,000 and total output={output_tokens:,} tokens/request: about ${estimate:.2f}.")

print("Game economics (simulation units, identical in all conditions):")
display(study.ECONOMICS)


## 2. Profit rules and experimental controls

| Economic setting | New value | Previous value |
|---|---:|---:|
| Revenue per actual customer delivery | 10 | No explicit revenue |
| Cost per newly committed factory unit | 2 | No explicit production cost |
| Holding cost per unit per week | 0.25 | 0.5 |
| Backlog penalty per unit per week | 2 | 1 |
| Initial inventory per role | 4 | 12 |
| Initial external demand per week | 8 | 4 |

**Team profit = retailer sales revenue − factory production commitments − all holding costs − all backlog penalties.** Each amount is counted once; internal supply-chain transfers do not create revenue or additional purchase costs. New factory units are paid for when ordered. Backorders remain outstanding and earn revenue only when eventually delivered; penalties already incurred remain. Initial inventory and pipeline are fixed sunk endowments. Unsold terminal stock/pipeline receive no salvage credit. These are illustrative simulation settings, not empirically calibrated student or market values.

All four treatment cells share exactly these economics and matched demand paths. Only the training lesson and reasoning effort differ. Training does not prescribe beta, an order quantity, or a desired hypothesis result. Structured answers still verify observed state; rules checks now also test profit arithmetic. Agents only see local observations and their own history.

The optional diagnostic stage now contains 24 retailer states per condition (96 orders + 16 checks), including shortage, balanced and surplus states. It is not selected by default and is still not a full game.


In [ ]:
print("SHARED RULES\n", study.STANDARD)
print("\nCONTROL LESSON\n", study.CONTROL)
print("\nTRAINING LESSON\n", study.TRAINING)


## 3. Run — the only cell that sends API requests

The progress bar counts completed responses, including rules checks, and shows how many remain. Requests run sequentially to limit budget exposure, but four game orders all use the same weekly snapshot before any are committed. Close or interrupt the cell to stop.

If the local spending guard stops, successful responses are cached; raising only `BUDGET_USD` allows resumption. If an actual request failed, preserve the failed pilot, fix the reported issue, and start a fresh replicate. Unknown failed-request usage is conservatively reserved. Changing settings is a new experiment, not a continuation of the old data.

In [ ]:
# Free estimator check: known beta must be recovered; copied demand must be rejected.
measurement_checks = study.measurement_self_check()
display(measurement_checks)
if not measurement_checks.passed.all():
    raise RuntimeError("Estimator self-check failed. Stop before running paid requests.")
print("Software check passed. This does not guarantee that GPT behavior will identify beta.")

# No GPT calls: check that service-oriented replenishment beats both extremes.
economic_checks = study.economic_sanity_check()
display(Markdown("### Free incentive check — synthetic policies, not GPT"))
display(economic_checks)
policies = economic_checks.set_index("policy")
assert policies.loc["replenish", "team_profit"] > policies.loc["never_order", "team_profit"]
assert policies.loc["replenish", "team_profit"] > policies.loc["order_100_each_week", "team_profit"]
print("Incentives verified on this fixed demand path. A sensible policy can still lose money; this is not an optimized-profit guarantee.")


In [ ]:
results, run_dir = None, None
client = None
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("An API key is required in API mode.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=REQUEST_TIMEOUT_S)
        del key
    results, run_dir = await study.run_study(BASE, CONFIG, client=client)
    print("Saved to:", run_dir)
finally:
    if client is not None:
        await client.close()


## 4. Conclusions and detailed outputs

**Diagnostics:** compare how orders change when supply line varies with inventory held fixed, and vice versa. These finite differences are sensitivity checks, not β estimates. Flat responses or extensive zero orders mean the structural β model may still be unsuitable. Do not choose prompts based on whether trained orders move in the desired direction.

**Game:** fit β only when identification checks pass. Preserve all fitted values and exclusion reasons. The report shows conclusions first; CSV files hold the details.

There are two planned comparisons: H1 averaged across effort, and H2_API low−high among trained groups. Analysis uses paired **blocks of four groups**, not individual weeks. Two-sided paired sign-randomization p-values assume exchangeability under the sharp null; Holm adjustment accounts for the two tests. No p-values are reported for offline data, fewer than 8 complete blocks, or after any planned block is lost to β nonidentification. This conservative rule prevents significance testing on a behavior-selected subset. The reporting floor of 8 is not a power justification.

The β model is unchanged from v1: forecast + α × (target − net inventory − β × supply line), clipped at zero. If α is near zero, β is not identifiable. Do not relax this check to get a p-value.

**Where to see the outputs:** the next cell displays condition performance and role-level β directly. All groups remain in performance summaries, including those without usable β. The run folder contains `condition_performance.csv`, `group_performance.csv`, `role_performance.csv`, `weekly_performance.csv`, `supply_chain_performance.png`, `agent_beta_fits.csv`, `group_beta.csv`, and `hypothesis_tests.csv`. β is not a performance score; higher β means greater estimated attention to outstanding orders, not necessarily lower cost.


In [ ]:
if results is None or run_dir is None:
    raise RuntimeError("Complete the run cell successfully before generating a report.")
report = study.make_report(results, run_dir, CONFIG)
display(Markdown(report))
print("Detailed files:")
for path in sorted(run_dir.glob("*.csv")):
    print(path.name)

if STAGE == "game":
    import pandas as pd
    from IPython.display import Image
    display(Markdown("### Supply-chain performance — all groups"))
    display(pd.read_csv(run_dir / "condition_performance.csv"))
    display(Image(filename=str(run_dir / "supply_chain_performance.png")))
    display(Markdown("### Role-level β — blank means not identifiable"))
    beta_results = pd.read_csv(run_dir / "agent_beta_fits.csv")
    display(beta_results[["agent_id", "trained", "effort", "alpha", "beta_for_interpretation", "usable", "fit_flag"]])
    display(Markdown("### H1 / H2_API test status"))
    display(pd.read_csv(run_dir / "hypothesis_tests.csv"))


## 5. Plan sample size before a confirmation run

Larger samples can improve power **after** reliable measurement is demonstrated. The table below uses a normal approximation for 80% power and conservative two-sided α=.025 per hypothesis. The standardized effect is a meaningful block contrast divided by its standard deviation, not a target picked to match this pilot. It ignores estimation failure and small-sample/randomization-test discreteness; use simulation-based planning with a stable pilot before treating a number as final.

Do not add groups repeatedly while checking p-values. Choose a fixed sample, analysis, prompts, model, and stopping rules beforehand. A significant result in this protocol would be evidence about these GPT agents under these conditions, not students.

In [ ]:
display(study.sample_size_scenarios())
